# Lab 1 — Supervised Fine-Tuning: contract review that cites its evidence

## Notebook 2 — Fine-tune with a SageMaker Training job

We now adapt **NVIDIA Nemotron 3 Nano 4B** to the checklist using **LoRA**, run as a
SageMaker AI **Training job**: a container, a script, and four S3 channels.

### Why a Training job and not serverless customization

The 30B-A3B version of this lab hands a JumpStart model id to `SFTTrainer` and lets
SageMaker pick the recipe, the container and the hyperparameter surface. Nemotron 3 Nano
4B is **not offered for serverless customization**, so that path is closed and we supply
those three things ourselves:

| | serverless customization | this notebook |
|---|---|---|
| model | JumpStart id | Hugging Face id, pulled from the Hub inside the job |
| training code | managed recipe | `scripts/train.py` (TRL `SFTTrainer` + PEFT) |
| configuration | `trainer.hyperparameters.*` | `args.yaml`, written below and mounted as a channel |
| container | managed | SageMaker PyTorch DLC, `image_uris.retrieve` |
| data | AI Registry `DataSet` entries | plain S3 prefixes as channels |
| output | registered Model Package | `model.tar.gz` in S3 |

More moving parts, and in exchange the whole configuration is a file you can read. The
trade worth naming is at the end: **there is no Model Package**, so the managed evaluators
the serverless lab uses have nothing to point at. Notebook 4 measures the model through a
deployed endpoint instead, which is why deployment comes before evaluation here.


In [ ]:
%load_ext autoreload
%autoreload 2

#### Setup and dependencies

The same boilerplate as notebook 1: execution role, default bucket, bucket prefix and the
boto3 clients. `role` is what the Training job runs as, and `bucket_name` /
`default_prefix` rebuild the data paths notebook 1 wrote to.


In [ ]:
import boto3
from sagemaker.core.helper.session_helper import Session, get_execution_role

sess = Session()
sagemaker_session_bucket = None
if sagemaker_session_bucket is None and sess is not None:
    sagemaker_session_bucket = sess.default_bucket()

try:
    role = get_execution_role()
except ValueError:
    iam = boto3.client("iam")
    role = iam.get_role(RoleName="sagemaker_execution_role")["Role"]["Arn"]

s3_client = boto3.client("s3")
sess = Session(default_bucket=sagemaker_session_bucket)
sm_client = boto3.client("sagemaker", region_name=sess.boto_region_name)
bucket_name = sess.default_bucket()
default_prefix = sess.default_bucket_prefix
region = sess.boto_region_name

print(f"sagemaker role arn: {role}")
print(f"sagemaker bucket: {bucket_name}")
print(f"sagemaker session region: {sess.boto_region_name}")


### Where the data is

Notebook 1 uploaded `train` and `val` and printed their URIs. Nothing was registered, so
this notebook rebuilds the same two paths from `config.py` rather than looking anything up
— which is also the check that both notebooks agree on where the data lives. If the
`head_object` calls below fail, notebook 1 has not been run against this bucket.

`TRAIN_JOB_PREFIX` comes from `config.py` too. It is the job's `base_job_name`, and
notebook 3 uses the same constant to find the finished job, so the two cannot drift.


In [ ]:
import os

from config import BASE_MODEL_ID, DATA_PREFIX, TRAIN_JOB_PREFIX

input_path = f"{default_prefix}/{DATA_PREFIX}" if default_prefix else DATA_PREFIX

train_dataset_s3_path = f"s3://{bucket_name}/{input_path}/train/dataset.jsonl"
val_dataset_s3_path = f"s3://{bucket_name}/{input_path}/val/dataset.jsonl"
train_config_s3_path = f"s3://{bucket_name}/{input_path}/config/args.yaml"

# Fail here, cheaply, rather than 3 minutes into a job that cannot find its data.
for uri in (train_dataset_s3_path, val_dataset_s3_path):
    key = uri.split(f"{bucket_name}/", 1)[1]
    size = s3_client.head_object(Bucket=bucket_name, Key=key)["ContentLength"]
    print(f"{uri}  ({size:,} bytes)")

print(f"\nbase model:      {BASE_MODEL_ID}")
print(f"job base name:   {TRAIN_JOB_PREFIX}")

# Exported so the %%bash cell below can substitute them into the recipe. A %%bash cell
# runs a real shell, which inherits this process's environment - it cannot see plain
# Python variables.
os.environ["model_id"] = BASE_MODEL_ID
os.environ["mlflow_uri"] = ""
os.environ["mlflow_experiment_name"] = "nemotron-3-nano-4b-contractnli-sft"


### The recipe

`train.py` parses its configuration with TRL's `TrlParser`, which reads one YAML file
holding both the script's own arguments and every `SFTConfig` field. The cell below writes
that file, and it is the whole training configuration — LoRA rank, learning rate, sequence
length, precision, which modules get adapters. It is then uploaded to its own `config`
channel, so SageMaker mounts it at `/opt/ml/input/data/config/args.yaml` — the single
hyperparameter the job is given.

`${model_id}`, `${mlflow_uri}` and `${mlflow_experiment_name}` are substituted by the shell
from the environment variables exported above, so the base model is set once in `config.py`
and flows through to the job.

Four settings are specific to this architecture and would be wrong if copied from a dense
model:

- **`target_modules`** — Nemotron-H is a Mamba2-Transformer hybrid: 21 Mamba2 layers, 4
  attention, 17 MLP, so only 4 of 42 layers have `q/k/v/o_proj` at all. `in_proj` **is**
  trainable — the mixer calls it as a module — but `out_proj` is **not**: the fused Mamba
  kernel reads `out_proj.weight` as a raw tensor, so an adapter there never receives a
  gradient. That is 21 dead adapters and a DDP *"parameters that didn't receive grad"*
  error; NVIDIA's own NeMo recipe for this family excludes it for the same reason. And
  `gate_proj` does not exist, because this MLP is ReLU² rather than gated, so a
  Llama-style list silently matches nothing for that entry.
- **`load_in_4bit: false`** — 4-bit is reported not to work on this architecture, and at
  7.95 GB of bf16 weights it buys nothing.
- **`pad_token: <unk>`** — the tokenizer defines no pad token and the usual fallback is
  eos. That is wrong under completion-only loss: the collator labels every pad position
  `-100`, so padding with `<|im_end|>` would mask the real turn terminator and the model
  would never learn to stop. `<unk>` is never emitted by this chat template.
- **`eos_token: <|im_end|>`** — `config.json` declares `eos_token_id: 2` (`</s>`), which
  this chat template never produces. Without the override the merged checkpoint ships with
  a stop token it cannot emit.


In [ ]:
%%bash

cat > ./args.yaml <<EOF
# ---------------------------------------------------------------- ScriptArguments
model_id: "${model_id}"
mlflow_uri: "${mlflow_uri}"
mlflow_experiment_name: "${mlflow_experiment_name}"

# Channel mount points: SageMaker copies each InputData channel to
# /opt/ml/input/data/<channel>/ before train.py runs.
train_dataset_path: "/opt/ml/input/data/train/dataset.jsonl"
val_dataset_path: "/opt/ml/input/data/val/dataset.jsonl"

# prompt + completion columns. TRL turns on completion-only loss by itself for this
# format, so the contract in the prompt is context and only the JSON is supervised.
dataset_format: "prompt_completion"

# trust_remote_code is deliberately unset: transformers implements nemotron_h natively, so
# train.py resolves it to the native class. NVIDIA's remote modeling code declares no
# _supports_flash_attn / _supports_sdpa flags (eager only, transformers #43352).
attn_implementation: "flash_attention_2"
torch_dtype: "bfloat16"
cast_parameters_to_uniform_dtype: false

# bf16 LoRA, not QLoRA.
load_in_4bit: false
use_peft: true
merge_weights: true

lora_r: 32
lora_alpha: 64
lora_dropout: 0.05

# Do NOT replace with "all-linear": it pulls in 21 dead Mamba out_proj adapters (the fused
# kernel takes out_proj.weight as a raw tensor, so they get no gradient) and matches
# nothing for gate_proj, which this ReLU^2 MLP does not have.
target_modules:
  - "q_proj"        # 4 attention layers
  - "k_proj"
  - "v_proj"
  - "o_proj"
  - "up_proj"       # 17 MLP layers
  - "down_proj"
  - "in_proj"       # 21 Mamba2 layers

# The tokenizer defines no pad token, and config.json's eos_token_id (2, </s>) is never
# emitted by the chat template.
pad_token: "<unk>"
eos_token: "<|im_end|>"

# Stop once eval_loss stops improving, rather than always running all 10 epochs. train.py
# turns this into an EarlyStoppingCallback(patience=3, threshold=0.01) and sets
# load_best_model_at_end / metric_for_best_model=eval_loss / greater_is_better=False.
# 423 records x 10 epochs is a lot of passes for what is largely a formatting task, so
# this cuts both cost and overfitting risk - and it picks the epoch count from the eval
# curve instead of guessing it.
early_stopping: true

# ---------------------------------------------------------------- SFTConfig
# /opt/ml/model is what SageMaker uploads to S3. With merge_weights above, the artifact is
# a full merged checkpoint vLLM can serve directly - no adapter to apply at load time.
output_dir: "/opt/ml/model"

# Measured with this model's own tokenizer in notebook 1. Records over it are truncated,
# not dropped, and the tail is short.
max_length: 8192

# packing must stay off: for prompt/completion records TRL derives the completion mask from
# the token-length difference, and packing destroys that boundary.
packing: false
completion_only_loss: true

num_train_epochs: 1
per_device_train_batch_size: 1
# Must be set explicitly: TrainingArguments defaults it to 8, independently of the train
# batch size, so leaving it out runs evaluation at 8x the batch that training uses. At
# max_length 8192 on a 22 GiB A10G that OOMs inside the first Mamba mixer of the first
# eval pass - after a full epoch of healthy training, which makes it look like a
# mid-training failure rather than an eval one.
per_device_eval_batch_size: 1
gradient_accumulation_steps: 4
learning_rate: 1.0e-4
# transformers 5.x merged warmup_ratio into warmup_steps: an int is a step count,
# a float in [0, 1) is a ratio of total training steps. `warmup_ratio` no longer
# exists, and TrlParser aborts on unknown config keys.
warmup_steps: 0.1
lr_scheduler_type: "cosine"
max_grad_norm: 1.0
weight_decay: 0.0

bf16: true
gradient_checkpointing: true

# Safe only because out_proj is excluded above - every adapter receives a gradient.
ddp_find_unused_parameters: false

logging_strategy: "steps"
logging_steps: 5
eval_strategy: "epoch"
# Must match eval_strategy: load_best_model_at_end (set by early_stopping above) raises
# "--load_best_model_at_end requires the save and eval strategy to match" otherwise. These
# are PEFT checkpoints, so each is the ~140 MB adapter rather than the full 8 GB model, and
# save_total_limit keeps only one on disk beside the best.
save_strategy: "epoch"
save_total_limit: 1
seed: 42
EOF

echo "wrote args.yaml"
cat ./args.yaml


In [ ]:
s3_client.upload_file("args.yaml", bucket_name, f"{input_path}/config/args.yaml")
os.remove("./args.yaml")

print(f"training config uploaded to:\n  {train_config_s3_path}")


### The training container

The SageMaker PyTorch Deep Learning Container, resolved for this region and instance type.
It ships PyTorch and CUDA; `scripts/requirements.txt` installs everything else — including
`transformers`, `trl` and `peft` at pinned versions — when the job starts.

`ml.g5.12xlarge` is 4x NVIDIA A10G (24 GB each). `Torchrun()` below makes that a 4-process
DDP job, so the effective batch is `per_device_train_batch_size x
gradient_accumulation_steps x 4`.


In [ ]:
from sagemaker.core import image_uris

instance_type = "ml.g5.2xlarge"
instance_count = 1

image_uri = image_uris.retrieve(
    framework="pytorch",
    region=region,
    version="2.8.0",
    instance_type=instance_type,
    image_scope="training",
)

print(image_uri)


### Configure the trainer

`ModelTrainer` is the Training-job equivalent of the serverless `SFTTrainer`, and the
pieces map one to one:

- **`SourceCode`** — the local `./scripts` directory is uploaded and unpacked in the
  container. `requirements.txt` is installed first, then `entry_script` runs.
- **`Compute`** — instance type and count, plus `keep_alive_period_in_seconds`. A non-zero
  value keeps the instance in a **warm pool** after the job ends, so a relaunch within that
  window skips both the capacity wait and the container pull. Set it to `0` to release the
  instance immediately.
- **`Torchrun()`** — launches one process per GPU, which is what makes this a DDP run.
- **`hyperparameters`** — a single entry, the in-container path to the recipe. Everything
  else lives in the YAML.
- **`OutputDataConfig`** — where the merged model is uploaded. Because `merge_weights: true`
  and `output_dir: /opt/ml/model`, the artifact is a **full merged checkpoint**, not an
  adapter, so vLLM can serve it directly in notebook 3.
- **`CheckpointConfig`** — mid-training checkpoints, synced to S3 so a spot interruption or
  a restart does not lose the run.

Note where the output lands: `s3://<bucket>/[<prefix>/]<TRAIN_JOB_PREFIX>/`, and SageMaker
appends `<full-job-name>/output/model.tar.gz` under it. Notebook 3 needs that artifact, and
it reads the exact URI from `DescribeTrainingJob` rather than rebuilding this string.


In [ ]:
from sagemaker.core.training.configs import (
    CheckpointConfig,
    Compute,
    OutputDataConfig,
    SourceCode,
    StoppingCondition,
)
from sagemaker.train.distributed import Torchrun
from sagemaker.train.model_trainer import ModelTrainer

source_code = SourceCode(
    source_dir="./scripts",
    requirements="requirements.txt",
    entry_script="train.py",
)

# Warm pool: hold the instance for 30 minutes after the job ends so a relaunch skips both
# the capacity wait and the container pull - together about 14 of the ~69 minutes this job
# takes. Worth it while iterating; the pool bills for as long as it is alive, so set this
# to 0 for a one-shot run or when handing the lab to students.
KEEP_ALIVE_SECONDS = 0

compute_configs = Compute(
    instance_type=instance_type,
    instance_count=instance_count,
    keep_alive_period_in_seconds=KEEP_ALIVE_SECONDS,
)

output_path = (f"s3://{bucket_name}/{default_prefix}/{TRAIN_JOB_PREFIX}"
               if default_prefix else f"s3://{bucket_name}/{TRAIN_JOB_PREFIX}")


# The OOM message above recommends this, and the numbers back it up: at the failure
# 3.97 GiB was "reserved but unallocated", i.e. lost to fragmentation. Mamba mixers
# allocate a few large short-lived tensors per layer (the fp32 gate upcast), which is
# exactly the pattern expandable_segments handles better than the default allocator.
# Set here rather than in args.yaml so it is in the process environment before torch
# initialises its CUDA allocator.
runtime_environment = {"PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True"}

model_trainer = ModelTrainer(
    training_image=image_uri,
    source_code=source_code,
    environment=runtime_environment,
    base_job_name=TRAIN_JOB_PREFIX,
    compute=compute_configs,
    distributed=Torchrun(),
    stopping_condition=StoppingCondition(max_runtime_in_seconds=7200),
    hyperparameters={
        # The only hyperparameter: the recipe's path inside the container. TrlParser
        # reads the rest from the YAML mounted on the `config` channel below.
        "config": "/opt/ml/input/data/config/args.yaml"
    },
    output_data_config=OutputDataConfig(s3_output_path=output_path),
    checkpoint_config=CheckpointConfig(
        s3_uri=output_path + "/checkpoint", local_path="/opt/ml/checkpoints"
    ),
    role=role,
    sagemaker_session=sess,
)

print(f"training output: {output_path}")

### The input channels

Each `InputData` becomes a directory in the container: `channel_name="train"` arrives at
`/opt/ml/input/data/train/`. Those paths are what `args.yaml` points `train_dataset_path`
and `val_dataset_path` at, and the `config` channel is where the recipe itself comes from.

This is the whole replacement for the AI Registry in the serverless lab — channels are
resolved by URI, so there is nothing to register and nothing to look up by name.


In [ ]:
from sagemaker.core.training.configs import InputData

data = [
    InputData(channel_name="train", data_source=train_dataset_s3_path),
    InputData(channel_name="val", data_source=val_dataset_s3_path),
    InputData(channel_name="config", data_source=train_config_s3_path),
]

for channel in data:
    print(f"{channel.channel_name:7s} -> /opt/ml/input/data/{channel.channel_name}/")
    print(f"{'':7s}    {channel.data_source}")


### Launch

`wait=False` returns immediately.

**How long, and where it goes.** At 423 records with an effective batch of 16 (4 GPUs x
batch 1 x 4 accumulation steps) an epoch is 27 optimizer steps, and a step measures about
11 s on `ml.g5.12xlarge` — so roughly 5 minutes per epoch. Around that sits a fixed cost
that is *not* training:

| phase | time |
|---|---|
| capacity wait (not billable) | varies, ~10 min |
| container pull | ~4 min |
| pip install + 8 GB model download | ~3.5 min |
| training | ~5 min per epoch |
| evaluation | ~25 s per epoch |
| merge + save + upload | ~8 min |

So a full 10 epochs is around **70 minutes billable**, of which ~16 are fixed overhead.
Halving the epochs does not halve the wall clock.

`early_stopping: true` in the recipe means the run usually stops before epoch 10: it ends
once `eval_loss` has failed to improve by 0.01 for 3 consecutive evaluations, then reloads
the best checkpoint. Watch the `eval_loss` column to see where it plateaued — that epoch
count is the one worth quoting when you rerun this deliberately.


In [ ]:
model_trainer.train(input_data_config=data, wait=False)

### What you built

A merged bf16 checkpoint in S3 — base weights with the LoRA adapter already applied, plus
the tokenizer — packaged as `model.tar.gz` under the job's output path.

Nothing was registered: no Model Package, no Model Package Group. The artifact *is* the
handoff, and notebook 3 finds it by asking `DescribeTrainingJob` for the last completed job
whose name starts with `TRAIN_JOB_PREFIX`.

Continue to **notebook 3** to deploy it to a SageMaker real-time endpoint. Evaluation is
notebook 4, after the endpoint exists, because without a Model Package there is no
managed evaluator to point at the model.
